### Data Cleaning and Quality Assessment
### Homework 1 – Tabular Data
### Hadi Seyed

### 1.2 Tidy Data from SQL

In [34]:
import sqlite3
import pandas as pd


DATABASE = "Film_Awards.sqlite"


def load_tables(database):
    connection = sqlite3.connect(database)

    actor = pd.read_sql_query(
        "SELECT * FROM actor",
        connection
    )

    award = pd.read_sql_query(
        "SELECT * FROM award",
        connection
    )
    
    director = pd.read_sql_query(
        "SELECT * FROM director",
        connection
    )

    org_name = pd.read_sql_query(
        "SELECT * FROM org_name",
        connection
    )

    connection.close()

    return actor, award, director, org_name


if __name__ == "__main__":
    actor, award, director, org_name = load_tables(DATABASE)
    
    
    print("Actor:")
    print(actor.head())

    print("\nAward:")
    print(award.head())

    print("\nDirector:")
    print(director.head())

    print("\nOrganization:")
    print(org_name.head())

Actor:
              name           film  year gender
0  Joaquin Phoenix          Joker  2019      M
1  Joaquin Phoenix  Walk the Line  2006      M
2  Joaquin Phoenix   Hotel Rwanda  2004      M
3  Joaquin Phoenix            Her  2013      M
4  Joaquin Phoenix     The Master  2013      M

Award:
  organization     award_name  year                                     winner
0        Oscar   Best Picture  2019                                   Parasite
1        Oscar   Best Picture  2018                                 Green Book
2        Oscar   Best Picture  2017                         The Shape of Water
3           GG  Picture/Drama  2017  Three Billboards Outside Ebbing, Missouri
4           GG  Picture/Drama  2018                          Bohemian Rhapsody

Director:
           name       film  year
0    Abel Gance  J'accuse!  1919
1    Abel Gance  J'accuse!  1938
2  Bong Joon-ho   Parasite  2019
3  Bong Joon-ho       Okja  2017
4  Bong Joon-ho    Sea Fog  2014

Organization:
   sh

In [27]:
# Only consider awards for the years 2017, 2018, and 2019.
award = award[
    award["year"].isin([2017, 2018, 2019])
]

actor = actor[
    actor["year"].isin([2017, 2018, 2019])
]

director = director[
    director["year"].isin([2017, 2018, 2019])
]

In [37]:
# Picture awards: winner is a film; director table supplies director.
picture = award[award["award_name"].str.contains("Picture|Drama|Comedy", regex=True)].copy()
picture = picture.merge(
    director,
    left_on=["winner","year"],
    right_on=["film","year"],
    how="left"
)
picture["Film"] = picture["winner"]
picture["Winner"] = pd.NA
picture["Director"] = picture["name"]
picture = picture[["year","award_name","organization","Winner","Film","Director"]]

# Acting awards: winner is a person; actor table supplies film.
acting = award[award["award_name"].str.contains("Actor|Actress", regex=True)].copy()
acting = acting.merge(
    actor,
    left_on=["winner","year"],
    right_on=["name","year"],
    how="left"
)
acting["Winner"] = acting["winner"]
acting["Director"] = pd.NA
acting = acting[["year","award_name","organization","Winner","film","Director"]]
acting = acting.rename(columns={"film":"Film"})

tidy_df = pd.concat([picture, acting], ignore_index=True)
tidy_df.columns = ["Year","Award","Organization","Winner","Film","Director"]
tidy_df = tidy_df.sort_values("Year", ascending=False).reset_index(drop=True)

# # Sort by year
# tidy_df = tidy_df.sort_values(
#     "Year",
#     ascending=False
# ).reset_index(drop=True)

# Save the tidy data to a new Excel file
output_file = "Film_Awards_SQL_Results.xlsx"

tidy_df.to_excel(
    output_file,
    index=False,
    sheet_name="Tidy Data"
)

print("\nTIDY SQL DATA")
print(tidy_df.to_string(index=False))

print(
    f"\nExcel file saved successfully as: {output_file}"
)


TIDY SQL DATA
 Year          Award Organization            Winner                                      Film           Director
 2019   Best Picture        Oscar               NaN                                  Parasite       Bong Joon-ho
 2019   Actor/Comedy           GG     Taron Egerton                                       NaN                NaN
 2019   Best Actress        Oscar   Renée Zellweger                                       NaN                NaN
 2019  Actress/Drama           GG   Renée Zellweger                                       NaN                NaN
 2019    Actor/Drama           GG   Joaquin Phoenix                                     Joker                NaN
 2019 Actress/Comedy           GG         Awkwafina                                       NaN                NaN
 2019     Best Actor        Oscar   Joaquin Phoenix                                     Joker                NaN
 2019 Actress/Comedy           GG               NaN                              

### Question 1: For each film involved in multiple awards, list the award and year.

In [29]:
film_awards = (
    award
    .dropna(subset=["winner"])
    .groupby("winner")
    .filter(lambda x: len(x) > 1)
)

film_awards = film_awards.sort_values(
    ["winner", "year"],
    ascending=[True, False]
)

print(
    film_awards[
        ["winner", "organization", "award_name", "year"]
    ].sort_values(["winner", "year"])
)

               winner organization      award_name  year
24  Frances McDormand        Oscar    Best Actress  2017
25  Frances McDormand           GG   Actress/Drama  2017
11        Gary Oldman        Oscar      Best Actor  2017
14        Gary Oldman           GG     Actor/Drama  2017
1          Green Book        Oscar    Best Picture  2018
7          Green Book           GG  Picture/Comedy  2018
9     Joaquin Phoenix        Oscar      Best Actor  2019
12    Joaquin Phoenix           GG     Actor/Drama  2019
21      Olivia Colman        Oscar    Best Actress  2018
22      Olivia Colman           GG  Actress/Comedy  2018
10         Rami Malek        Oscar      Best Actor  2018
13         Rami Malek           GG     Actor/Drama  2018
18    Renée Zellweger        Oscar    Best Actress  2019
19    Renée Zellweger           GG   Actress/Drama  2019


### Question 2: For each actor/actress winning multiple awards, list the film and award.

In [31]:
actor_awards = award.merge(
    actor,
    left_on=["winner", "year"],
    right_on=["name", "year"],
    how="inner"
)

actor_awards = actor_awards[
    actor_awards["year"].isin([2017, 2018, 2019])
]

actor_awards = actor_awards.sort_values(
    ["winner", "year"]
)

print(
    actor_awards[
        [
            "winner",
            "film",
            "organization",
            "award_name",
            "year"
        ]
    ]
)

              winner                                       film organization  \
2  Frances McDormand  Three Billboards Outside Ebbing, Missouri        Oscar   
3  Frances McDormand  Three Billboards Outside Ebbing, Missouri           GG   
0    Joaquin Phoenix                                      Joker        Oscar   
1    Joaquin Phoenix                                      Joker           GG   

      award_name  year  
2   Best Actress  2017  
3  Actress/Drama  2017  
0     Best Actor  2019  
1    Actor/Drama  2019  
